<a href="https://colab.research.google.com/github/ivanduzunov/AI-Agents-and-Workflows-for-Developers/blob/main/AI_AGENTS_LangChain_Agents_%26_Tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters

In [ ]:
# Agent
# Model
# Tools
# Loader -> load documents
# Splitter ->
# Retriever
# Middleware
# Chekpoinnter
# Store
# Vector Store
# Graph + Node


In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.messages import BaseMessage
from langchain_core.tools import create_retriever_tool
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_chroma import Chroma
from langchain.tools import tool
from langchain.agents.middleware.types import after_model
from langchain.agents import AgentState
from langgraph.runtime import Runtime
from google.colab import userdata
from pydantic import SecretStr
from typing import List
from IPython.display import Image
import json

In [ ]:

def print_conversation(messages: List[BaseMessage]):
  for message in messages:
    message.pretty_print()

In [ ]:
api_key = SecretStr(userdata.get('OPENAI_KEY'))
model = ChatOpenAI(model_name="gpt-5-nano", api_key=api_key, use_responses_api=True)

In [ ]:
text_loader = TextLoader("/content/company_faq_sample_questions.md")
documents = text_loader.load()

In [ ]:
splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "Header 1"), ("##", "Header 2")], strip_headers=False)


In [ ]:
chunks_to_embed = []
for document in documents:
  chunks_to_embed.extend(splitter.split_text(document.page_content))

In [ ]:
chunks_to_embed

In [ ]:
chroma_db = Chroma(collection_name="faq", persist_directory="/content/faq_chroma_db")


In [ ]:
chroma_db.add_documents(documents=chunks_to_embed)

In [ ]:
chroma_db.search(query="customer", search_type="similarity")

In [ ]:
# What now?
# 2. Transform the VDB -> retriever -> tool
chroma_retriever = chroma_db.as_retriever(search_kwargs={"k": 3})
search_knowledgebase = create_retriever_tool(
    retriever=chroma_retriever,
    name="search_knowledgebase",
    description="Call this tool to search in the internal knowledgebase using a natural language query",
    document_separator="\n\n\n ------------ \n\n\n"
)

# What now?
# 1. Implement a custom tool that accepts query, searches in the VDB, and returns results

# @tool
# def search_knowledgebase(query: str) -> str:
#   """
#   Call this tool to search in the internal knowledgebase
#   using a natural language query
#   """
#   results = chroma_db.similarity_search(query=query, k=3)

#   return "\n\n\n ------------ \n\n\n".join([document.page_content for document in results])

In [ ]:
@tool
def my_employees() -> str:
  """
  This tool will retrieve all the employees if asked for them.
  """

  employees = [{"id": 1, "name": "Keanu Reeves"}, {"id": 2, "name": "Arnold Schwarzenegger"}]

  return json.dumps(employees)


@tool
def lookup_employee_details(id: int) -> str:
  """
  This tool will retrieve additional details about given employee. Return them everytime when information for the employees is required.
  """

  employees = [{"id": 1, "name": "Keanu Reeves"}, {"id": 2, "name": "Arnold Schwarzenegger"}]

  details = {
      1: "The Matrix actor",
      2: "The Terminator actor",
  }

  result = details.get(int(id), "Unknown Employee")

  return result

In [ ]:
@after_model
def after_model_func(state: AgentState, runtime: Runtime):
  print("AFTER MODEL FUNC")

In [ ]:
agent = create_agent(
    debug=True,
    tools=[search_knowledgebase, my_employees, lookup_employee_details],
    middleware=[after_model_func],
    model=model,
    system_prompt="You are helpful customer support agent. When answering questions, use the search_knowledgebase tool whenever the answer could be found in the external knowledge base. Do not make up information that could be retrieved from the knowledge base."
)

In [ ]:
# RAG - Retrieval Augmented Genration
# CAG - Cache Augmented Gneration

In [ ]:
# CODE SNIPPETS

In [ ]:
from google.colab import userdata
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import SecretStr

def print_response(response: AIMessage):
  print(f"Response id: {response.id}")
  if response.usage_metadata is not None:
    input_tokens = response.usage_metadata.get("input_tokens", 0)
    cached_tokens = response.usage_metadata.get("input_token_details", {}).get("cache_read", 0)
    output_tokens = response.usage_metadata.get("output_tokens", 0)
    reasoning_tokens = response.usage_metadata.get("input_token_details", {}).get("reasoning", 0)

    print(f"Input tokens: {input_tokens}")
    print(f"Cached tokens: {cached_tokens}")
    print(f"Output tokens: {output_tokens}")
    print(f"Reasoning tokens: {reasoning_tokens}")




In [ ]:
api_key = SecretStr(userdata.get('OPENAI_KEY'))

openai_model = ChatOpenAI(
    model_name="gpt-5-nano",
    openai_api_key=api_key,
    reasoning_effort="low")

messages = [
    SystemMessage("You are a history teacher. And you have to answer questions."),
    HumanMessage("Write me the 20 most impoetant dates in the history of Ancient Rome.")
]

response = openai_model.invoke(input=messages)

print_response(response=response)


In [ ]:
first_response = openai_model.invoke(input=[HumanMessage("Hi, Im Ivan from Bulgaria. How are you today?")])


In [ ]:
second_response = openai_model.invoke(input=[
    HumanMessage("Hi, Im Ivan from Bulgaria. How are you today?"),
    first_response,
    HumanMessage("What do you remember about me?")
    ])
print(second_response)


In [ ]:
from google.colab import userdata
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from pydantic import SecretStr
from typing import List

def print_conversation(conversation: List[AIMessage]):
  """
  Getting the conversation and printing it of the database status.
  """
  for message in conversation:
    message.pretty_print()


In [ ]:
@tool
def get_database_status() -> str:
  """
  Getting the status of the database.
  """
  return "Its Ok!"

In [ ]:
api_key = SecretStr(userdata.get('OPENAI_KEY'))

openai_model = ChatOpenAI(model_name="gpt-5-nano", api_key=api_key).bind_tools([get_database_status])


In [ ]:
first_response = openai_model.invoke(input=[
    HumanMessage("What is the current status of the database??"),
])



In [ ]:
print_conversation([first_response])

In [ ]:
status = get_database_status.invoke("")
print(status)

In [ ]:
tool_call_answer = ToolMessage(status, tool_call_id=first_response.tool_calls[0]["id"])